In [0]:
print("Hello World")

In [0]:
print("Hello")

In [0]:
%sql
select * from samples.nyctaxi.trips;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# 1. method - without strict schema enforcement
stu_data = [
    (100, 'Pankaj', 'ECE'),
    (101, 'Manoj', 'EEE'),
    (102, 'Babitha', 'CSE'),
    (103, 'Reshma', 'ECE'),
]

columns = ["stu_id", "stu_name", 'branch']

stu_df = spark.createDataFrame(data=stu_data, schema=columns)
stu_df.show()  # -- action cmd

In [0]:
display(stu_df)

Databricks data profile. Run in Databricks to view.

In [0]:
# 2. method - with strict schema enforcement
stu_data = [
    (100, 'Pankaj', 'ECE'),
    (101, 'Manoj', 'EEE'),
    (102, 'Babitha', 'CSE'),
    (103, 'Reshma', 'ECE'),
]

columns = T.StructType([
    T.StructField("stu_id", T.IntegerType(), False),
    T.StructField("stu_name", T.StringType(), True),
    T.StructField("branch", T.StringType(), True),
])
            

stu_df = spark.createDataFrame(data=stu_data, schema=columns)
stu_df.show()  # -- action cmd

In [0]:
nyc_taxi_df = spark.sql("select * from samples.nyctaxi.trips where trip_distance > 1")
display(nyc_taxi_df)

In [0]:
nyc_taxi_df = spark.table("samples.nyctaxi.trips")
display(nyc_taxi_df)

In [0]:
elec_df = (
    spark.read.format("csv")
    .load("/FileStore/tables/samples/bihar_election_results.csv")
)
display(elec_df)

In [0]:
elec_df = (
    spark.read.format("csv")
    .option("header", "true")
    .load("/FileStore/tables/samples/bihar_election_results.csv")
)
display(elec_df)

In [0]:
elec_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/FileStore/tables/samples/bihar_election_results.csv")
)
display(elec_df)

In [0]:
# 1. method
elec_df2 = elec_df.select("Constituency Number", "Constituency Name")
elec_df2.display()

In [0]:
# 2. method
elec_df2 = elec_df.select(
    elec_df["Constituency Number"],
    elec_df["Constituency Name"]
)
elec_df2.display()

In [0]:
# 3. method
elec_df2 = elec_df.select(
    elec_df.Party
)
elec_df2.display()

In [0]:
# 4. method
elec_df2 = elec_df.select(
    F.col("Constituency Number"),
    F.col("Constituency Name")
)
elec_df2.display()

In [0]:
# 1. method -- filter

elec_df_1 = elec_df.filter(F.col("Constituency Name") == "VALMIKI NAGAR")
elec_df_1.display()

In [0]:
# 2. method -- where

elec_df_1 = elec_df.where(F.col("Constituency Name") == "VALMIKI NAGAR")
elec_df_1.display()

In [0]:
# collect  -- action
data = elec_df_1.collect()
data

In [0]:
len(data)

In [0]:
data[0]['Party']

In [0]:
for partyName in range(len(data)):
    print(data[partyName]['Party'])

In [0]:
# withColumn -- transformation
elec_df_1 = elec_df_1.withColumn("Country", F.lit("IND"))
elec_df_1.display()

In [0]:
elec_df_2 = (
    elec_df_1
    .withColumn("Country", F.lit("IND"))
    .withColumn("Test1", F.lit("dummy"))
    .withColumn("Test2", F.lit("dummy"))
)
elec_df_2.display()

In [0]:
# withColumnRenamed  -- trabsformation
elec_df_1 = elec_df_1.withColumnRenamed("Constituency Name", "ConstituencyName")
elec_df_1.display()

In [0]:
elec_df_1 = (
    elec_df_1
    .withColumnRenamed("Serial Number", "SerialNumber")
    .withColumnRenamed("Constituency Number", "ConstituencyNumber")
    .withColumnRenamed("Candidate Name", "CandidateName")
)
elec_df_1.display()

In [0]:
df = (
    elec_df_1
    .select(
        F.col("EVM Votes").alias("EVMVotes"),
        F.col("Postal Votes").alias("PostalVotes"),
        F.col("Total Votes").alias("TotalVotes")
    )
)

df.display()

In [0]:
elec_df_1.columns

In [0]:
for colName in elec_df_1.columns:
    newColName = colName.replace(" ", "")
    elec_df_1 = elec_df_1.withColumnRenamed(colName, newColName)
elec_df_1.display()

In [0]:
newSchema = [oldCol.replace(" ", "") for oldCol in elec_df_1.columns]
elec_df_1 = elec_df_1.toDF(*newSchema)
elec_df_1.display()

In [0]:
# distinct -- action
elec_df_1 = elec_df_1.distinct()
elec_df_1.display()

In [0]:
# dropDuplicates -- transformation
elec_df_1 = elec_df_1.dropDuplicates()
elec_df_1.display()

In [0]:
df2 = elec_df_1.dropDuplicates(subset=['ConstituencyNumber', 'ConstituencyName'])
df2.display()

In [0]:
# orderBy & sort
display(
    elec_df_1
    .orderBy(
        F.col("TotalVotes").desc()
    )
)

In [0]:
# orderBy & sort
display(
    elec_df_1
    .sort(
        F.col("TotalVotes").desc()
    )
)

In [0]:
# group by 
stats_df = (
    elec_df
    .groupBy("Party")
    .count()
)
(stats_df.orderBy(F.col("count").desc())).display()

In [0]:
# group by 
stats_df = (
    elec_df
    .groupBy("Party")
    .agg(
        F.count("*").alias("TotalCadidatesPerParty"),
        F.sum("Total Votes").alias("TotalVotesPerParty")
    )
    .sort(F.col("TotalCadidatesPerParty").desc())
)
stats_df.display()